

> GPU T4확인




In [ ]:
!nvidia-smi



> Unsloth 설치



In [2]:
!pip install -q unsloth



> 모델 로드



In [3]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-14B-unsloth-bnb-4bit",
    max_seq_length=4096,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.2: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]



> 메모리 확인





In [ ]:
import torch

used = torch.cuda.memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"{used:.1f}GB / {total:.1f}GB")



> QLoRA



In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)



> 학습된 LORA 붙이기




In [4]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    model,
    "/content/drive/MyDrive/qwen3-finetune/lora_model",
)
FastLanguageModel.for_inference(model)

print(type(model))
print(model.peft_config)

<class 'peft.peft_model.PeftModelForCausalLM'>
{'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping={'base_model_class': 'Qwen3ForCausalLM', 'parent_library': 'transformers.models.qwen3.modeling_qwen3', 'unsloth_fixed': True}, peft_version='0.21.1', base_model_name_or_path='unsloth/Qwen3-14B-unsloth-bnb-4bit', revision=None, inference_mode=True, r=16, target_modules={'q_proj', 'k_proj', 'down_proj', 'o_proj', 'up_proj', 'gate_proj', 'v_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_con



> 데이터 로드



In [ ]:
from datasets import load_dataset

WORK_DIR = "/content/drive/MyDrive/qwen3-finetune"

dataset = load_dataset("json", data_files=f"{WORK_DIR}/train.jsonl", split="train")

def formatting(examples):
    texts = [
        tokenizer.apply_chat_template(m, tokenize=False)
        for m in examples["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(formatting, batched=True)
print(dataset)



> 길이 확인



In [ ]:
lengths = [len(tokenizer(t)["input_ids"]) for t in dataset["text"]]
print(f"최소 {min(lengths)} / 평균 {sum(lengths)//len(lengths)} / 최대 {max(lengths)}")



> 학습 설정





In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=4096,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=True,
        bf16=False,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=f"{WORK_DIR}/outputs",
        save_steps=20,
        save_total_limit=3,
        report_to="none",
    ),
)



> 학습 시작



In [ ]:
trainer.train()

model.save_pretrained(f"{WORK_DIR}/lora_model")
tokenizer.save_pretrained(f"{WORK_DIR}/lora_model")
print("저장 완료")



> 파싱 성공률 측정



In [7]:
import json

with open("/content/drive/MyDrive/qwen3-finetune/val.jsonl") as f:
    val = [json.loads(line) for line in f]

ok_count = 0
for i, sample in enumerate(val[:30]):
    prompt = sample["messages"][0]["content"]
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(input_ids=inputs, max_new_tokens=3072, temperature=0.7)
    text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

    try:
        parsed = json.loads(text)
        ok_count += 1
        print(f"{i+1:2d}. 성공 {parsed.get('problem_title', '')}")
    except Exception:
        print(f"{i+1:2d}. 실패 {text[:40]}")

print()
print(f"파싱 성공률 {ok_count}/10")

Both `max_new_tokens` (=3072) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=3072) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 1. 성공 최대 차이 구간


Both `max_new_tokens` (=3072) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 2. 성공 직선으로 정사각형 나누기


Both `max_new_tokens` (=3072) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

27. 성공 트리 위의 최댓값


Both `max_new_tokens` (=3072) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


28. 성공 문자열의 등장 페어


Both `max_new_tokens` (=3072) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


29. 성공 정수의 순환 합
30. 성공 인접 문자 제거

파싱 성공률 29/10




> 잘림 확인(실패한 6번 재생성)



In [6]:
import json

with open("/content/drive/MyDrive/qwen3-finetune/val.jsonl") as f:
    val = [json.loads(line) for line in f]

sample = val[5]
prompt = sample["messages"][0]["content"]
inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=3072, temperature=0.7)
text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print(f"길이 {len(outputs[0]) - inputs.shape[1]} 토큰")
print(text[-300:])

Both `max_new_tokens` (=3072) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


길이 741 토큰
s": [{"language": "PYTHON", "time_limit_ms": 2000, "memory_limit_kb": 262144}, {"language": "JAVA", "time_limit_ms": 2000, "memory_limit_kb": 262144}, {"language": "JAVASCRIPT", "time_limit_ms": 2000, "memory_limit_kb": 262144}, {"language": "CPP", "time_limit_ms": 2000, "memory_limit_kb": 262144}]}




> 구글 드라이브 연결



In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

WORK_DIR = "/content/drive/MyDrive/qwen3-finetune"
os.makedirs(WORK_DIR, exist_ok=True)
print(WORK_DIR)

In [ ]:
!ls -la /content/drive/MyDrive/qwen3-finetune/

In [ ]:
print(model.peft_config)



> 측정 결과 저장



In [9]:
import json
from datetime import datetime

result = {
    "날짜": datetime.now().isoformat(),
    "모델": "Qwen3-14B-unsloth-bnb-4bit",
    "lora_r": 16,
    "학습데이터": 270,
    "에폭": 2,
    "loss_시작": 1.331,
    "loss_끝": 0.420,
    "파싱성공": 29,
    "파싱전체": 30,
    "비고": "val.jsonl 30건, temperature 0.7, max_new_tokens 3072",
}

path = "/content/drive/MyDrive/qwen3-finetune/result_v1.json"
with open(path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"저장: {path}")
print(json.dumps(result, ensure_ascii=False, indent=2))

저장: /content/drive/MyDrive/qwen3-finetune/result_v1.json
{
  "날짜": "2026-10-08T04:22:34.961158",
  "모델": "Qwen3-14B-unsloth-bnb-4bit",
  "lora_r": 16,
  "학습데이터": 270,
  "에폭": 2,
  "loss_시작": 1.331,
  "loss_끝": 0.42,
  "파싱성공": 29,
  "파싱전체": 30,
  "비고": "val.jsonl 30건, temperature 0.7, max_new_tokens 3072"
}
